In [1]:
# =========================================================
# STEP 4 — TOPIC MODELING & PUBLIC ISSUE DETECTION
# FILE: notebooks/04_topic_modeling.ipynb
# =========================================================

# =========================================================
# STEP 4.1 — IMPORT LIBRARY
# =========================================================

import pandas as pd
import numpy as np
import warnings

import plotly.express as px

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")


# =========================================================
# STEP 4.2 — LOAD CLEAN DATASET
# =========================================================

DATA_PATH = "../data/processed/clean_public_sentiment.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
df.head()


# =========================================================
# STEP 4.3 — PREPARE TEXT DATA
# =========================================================

documents = df["clean_comment"].dropna().astype(str)

print("Number of documents:", len(documents))


# =========================================================
# STEP 4.4 — COUNT VECTORIZATION
# =========================================================

vectorizer = CountVectorizer(
    max_features=1000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

doc_term_matrix = vectorizer.fit_transform(documents)

print("Document-Term Matrix Shape:", doc_term_matrix.shape)


# =========================================================
# STEP 4.5 — TOP KEYWORDS (GLOBAL)
# =========================================================

term_frequencies = np.asarray(
    doc_term_matrix.sum(axis=0)
).flatten()

terms = vectorizer.get_feature_names_out()

keyword_df = pd.DataFrame({
    "keyword": terms,
    "frequency": term_frequencies
}).sort_values(
    "frequency",
    ascending=False
).reset_index(drop=True)

top_keywords = keyword_df.head(20)

top_keywords

Dataset Shape: (5000, 19)
Number of documents: 5000
Document-Term Matrix Shape: (5000, 24)


,keyword,frequency
0,informasipublikjadilebihtransparan,382
1,programinisangatmembantumasyarakat,367
2,salutuntukinovasidigitalkotabatu,360
3,pelayananonlinesekaranglebihefisien,353
4,digitalisasilayanansangatmempermudahwarga,343
5,pelayanansekarangjauhlebihcepatdanmudah,334
6,aplikasipemerintahmakinmoderndanresponsif,332
7,wifipubliksangatmembantumahasiswa,297
8,masihperlupengembanganlebihlanjut,203
9,semogalayananterusditingkatkan,200


In [2]:
# =========================================================
# STEP 4.6 — VISUALIZE TOP KEYWORDS
# =========================================================

fig = px.bar(
    top_keywords,
    x="frequency",
    y="keyword",
    orientation="h",
    title="Top Public Issue Keywords",
    template="plotly_dark",
    color="frequency",
    color_continuous_scale="Viridis"
)

fig.update_layout(
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    font_color="white",
    title_font_size=24,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

# Simpan screenshot sebagai:
# assets/topic-keywords.png

In [3]:
# =========================================================
# STEP 4.7 — TOPIC MODELING WITH LDA
# =========================================================

n_topics = 5

lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch"
)

lda.fit(doc_term_matrix)

print("LDA Topic Modeling Completed.")

LDA Topic Modeling Completed.


In [4]:
# =========================================================
# STEP 4.8 — EXTRACT TOP WORDS PER TOPIC
# =========================================================

def get_top_words(model, feature_names, n_top_words=10):
    topics = []

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_terms = [feature_names[i] for i in top_indices]

        topics.append({
            "Topic": f"Topic {topic_idx + 1}",
            "Top Keywords": ", ".join(top_terms)
        })

    return pd.DataFrame(topics)

topic_df = get_top_words(
    lda,
    vectorizer.get_feature_names_out(),
    n_top_words=10
)

topic_df

,Topic,Top Keywords
0,Topic 1,"pelayananonlinesekaranglebihefisien, semogalay..."
1,Topic 2,"programinisangatmembantumasyarakat, fiturnyacu..."
2,Topic 3,"informasipublikjadilebihtransparan, pelayanans..."
3,Topic 4,"salutuntukinovasidigitalkotabatu, aplikasipeme..."
4,Topic 5,"digitalisasilayanansangatmempermudahwarga, mas..."


In [5]:
# =========================================================
# STEP 4.9 — TOPIC IMPORTANCE
# =========================================================

topic_weights = lda.transform(doc_term_matrix).mean(axis=0)

topic_importance = pd.DataFrame({
    "Topic": [f"Topic {i+1}" for i in range(n_topics)],
    "Importance": topic_weights
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

topic_importance

,Topic,Importance
0,Topic 3,0.2103
1,Topic 4,0.2057
2,Topic 2,0.2009
3,Topic 1,0.1923
4,Topic 5,0.1908


In [6]:
# =========================================================
# STEP 4.10 — VISUALIZE TOPIC IMPORTANCE
# =========================================================

fig = px.bar(
    topic_importance,
    x="Topic",
    y="Importance",
    color="Importance",
    text="Importance",
    title="Government Public Issue Topic Importance",
    template="plotly_dark",
    color_continuous_scale="Plasma"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    font_color="white",
    title_font_size=24,
    yaxis_range=[0, topic_importance["Importance"].max() * 1.2]
)

fig.show()

# Simpan screenshot sebagai:
# assets/topic-modeling.png

In [7]:
# =========================================================
# STEP 4.11 — ISSUE SEVERITY ANALYSIS
# =========================================================

issue_severity = (
    df.groupby("urgency_level")
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

issue_severity

,urgency_level,count
1,Low,2768
2,Medium,1848
0,High,384


In [8]:
# =========================================================
# STEP 4.12 — VISUALIZE ISSUE SEVERITY
# =========================================================

fig = px.pie(
    issue_severity,
    names="urgency_level",
    values="count",
    hole=0.65,
    title="Issue Severity Monitoring",
    template="plotly_dark",
    color_discrete_map={
        "Low": "#22C55E",
        "Medium": "#F59E0B",
        "High": "#EF4444"
    }
)

fig.update_layout(
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    font_color="white",
    title_font_size=24
)

fig.show()

# Simpan screenshot sebagai:
# assets/issue-severity-analysis.png

In [9]:
# =========================================================
# STEP 4.13 — EXECUTIVE ISSUE SUMMARY
# =========================================================

top_issue = topic_importance.loc[0, "Topic"]

print(f"""
============================================================
EXECUTIVE ISSUE SUMMARY
============================================================

Dominant Topic : {top_issue}

Strategic Insight:
Topic modeling reveals the primary themes discussed by
the public regarding government digital programs. These
topics represent the most influential issues affecting
citizen perception and engagement, and can be used to
prioritize communication strategies and service improvements.

============================================================
""")


EXECUTIVE ISSUE SUMMARY

Dominant Topic : Topic 3

Strategic Insight:
Topic modeling reveals the primary themes discussed by
the public regarding government digital programs. These
topics represent the most influential issues affecting
citizen perception and engagement, and can be used to
prioritize communication strategies and service improvements.


